# 🎤 ISOM 260: Run a Focus Group Before Lunch

**Session 3 — Bonus Workshop: Your Idea × 25 Customers × 40 Seconds** | Suffolk University | Prof. Hasan Arslan

---

A real focus group costs **$5,000–15,000**, takes three weeks to schedule, and gives you eight people who are being polite.

In the next 15 minutes you will pitch **your own idea** to 25 synthetic customers, hear every one of them react in their own voice, see a chart of why they'd say no, fix your pitch, and re-run the room to see if the number moves. Then we'll be honest about what you just did and didn't learn.

| Part | What happens | The business skill |
|------|--------------|--------------------|
| 1 | One sentence: your idea | Writing a pitch someone can react to |
| 2 | Gemini invents 25 diverse customers (one call) | Segmentation — who is this even for? |
| 3 | All 25 react, structured, plus a chart (one call) | Reading objections instead of compliments |
| 4 | Rewrite the pitch, re-run the same 25 | The A/B loop every founder runs |
| 5 | The honesty check + the CFO check | Knowing what synthetic data can and can't tell you |
| **B** | **Press play: the model runs the loop itself on YOUR pitch** | Your first agent — and the audit that catches it cheating |
| 6 | *(optional, whole class)* The room's ideas become a chart | Your final project starts here |

**Setup:** same key as Wednesday in the 🔑 Secrets panel as `GOOGLE_API_KEY`. `File → Save a copy in Drive`. Total: **~12 API calls** including the agent, free tier, $0.

In [ ]:
# ── Setup (same as Wednesday) ─────────────────────────────────────
!pip install -q google-genai

In [ ]:
# ── Key + client + a helper that forces structured JSON ──────────
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal
import pandas as pd, json, time

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-2.5-flash"

def structured(prompt, schema, temperature=0.7):
    '''One call that must return JSON matching `schema`. Thinking is off:
    role-play doesn't need it, and it keeps the focus group under a cent.'''
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=schema,
        temperature=temperature, thinking_config=types.ThinkingConfig(thinking_budget=0))
    for attempt in range(4):                      # free-tier rate limits: wait and retry, don't crash
        try:
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            return r.parsed, r.usage_metadata
        except Exception as e:
            if attempt == 3 or ("429" not in str(e) and "RESOURCE_EXHAUSTED" not in str(e)): raise
            print(f"   ⏳ rate-limited, waiting {15*(attempt+1)}s…"); time.sleep(15*(attempt+1))

spent = []                    # we'll add up every call for the CFO check
print("✅ The room is ready. Bring an idea.")

## 1 · Your idea, in one sentence

Replace the sample below with **your** idea — a startup, a side hustle, a feature for the company you interned at, a club fundraiser, anything someone would have to *choose* to pay for or use. Be specific: who it's for, what it does, what it costs.

*(No idea in mind? Run the sample first — it's a real problem on this campus — and swap yours in on the second pass.)*

In [ ]:
# ── Your pitch ───────────────────────────────────────────────────
PITCH = (
    "MeterMate: a $6/month app for Boston college students that texts you 15 minutes "
    "before your street-parking meter expires and lets you extend it with one tap, "
    "so you never eat a $40 ticket during a two-hour lecture again."
)
TARGET = "college students in Boston who drive to campus at least twice a week"

print("🎤 PITCH:", PITCH, "\n👥 TARGET:", TARGET)

## 2 · Twenty-five customers walk into a room

You can't ask "everyone" — you have to decide *who*. Watch the model invent a deliberately mixed room: different budgets, different levels of skepticism, different lives. The `Literal` fields force the spread; the quirk field is what makes them feel like people.

In [ ]:
# ── Generate the room (one call) ─────────────────────────────────
class Persona(BaseModel):
    name: str
    age: int
    who_they_are: str = Field(description="Occupation/situation in under 10 words")
    monthly_fun_budget_usd: int = Field(description="Discretionary money per month, realistic for who they are")
    skepticism: Literal["easy_sell", "needs_convincing", "hard_no_by_default"]
    quirk: str = Field(description="One specific habit or belief that shapes how they buy, under 12 words")

N = 25
personas, u = structured(
    f'''Invent {N} realistic, DIVERSE potential customers for this idea.
Target audience: {TARGET}
Idea: {PITCH}
Requirements: a real spread of ages, budgets, skepticism levels, and life situations.
Include a few people just OUTSIDE the target audience — real markets always have them.
Make the quirks specific and different from each other. No duplicates.''',
    schema=list[Persona], temperature=1.0)
spent.append(u)

room = pd.DataFrame([p.model_dump() for p in personas])
print(f"✅ {len(room)} people in the room  ·  skepticism mix: {room.skepticism.value_counts().to_dict()}\n")
display(room)

## 3 · Pitch the room

Every persona reacts **in character**. The schema does the hard thing a human moderator struggles with: it forces each reaction into a *decision* (`would_pay`), a *number* (`price_they_would_pay`), and a *reason* (`top_objection`) — so the compliments can't hide the pattern.

In [ ]:
# ── The focus group (one call) ───────────────────────────────────
class Reaction(BaseModel):
    name: str
    would_pay: bool
    price_they_would_pay_per_month: float = Field(description="0 if they would not pay at all")
    top_objection: Literal["price", "dont_have_the_problem", "trust_or_privacy", "already_solved_another_way",
                           "too_much_effort", "doesnt_work_for_my_situation", "no_objection"]
    reaction: str = Field(description="One or two sentences, IN THEIR OWN VOICE, honest, not polite. Under 35 words.")
    what_would_change_their_mind: str = Field(description="Under 15 words. 'nothing' is allowed.")

def focus_group(pitch, personas, label):
    people = "\n".join(f"- {p.name}, {p.age}, {p.who_they_are}. Budget ${p.monthly_fun_budget_usd}/mo. "
                       f"Skepticism: {p.skepticism}. Quirk: {p.quirk}" for p in personas)
    reactions, u = structured(
        f'''You are running a focus group. Each person below hears this pitch and reacts as THEMSELVES —
consistent with their budget, skepticism and quirk. Skeptics stay skeptical unless genuinely convinced.
Return exactly one reaction per person, in order.

PITCH: {pitch}

PEOPLE:
{people}''',
        schema=list[Reaction], temperature=0.8)
    spent.append(u)
    df = pd.DataFrame([r.model_dump() for r in reactions])
    df["round"] = label
    return df

r1 = focus_group(PITCH, personas, "v1")
rate1 = r1.would_pay.mean()
print(f"💬 {len(r1)} reactions back.  Would pay: {r1.would_pay.sum()}/{len(r1)}  ({rate1:.0%})")
print(f"   Median price among the yeses: ${r1[r1.would_pay].price_they_would_pay_per_month.median():.2f}/mo\n")
for _, r in r1.head(6).iterrows():
    print(f"{'✅' if r.would_pay else '❌'} {r['name']}: “{r.reaction}”")
print("   …")

In [ ]:
# ── Why they said no (the chart that matters) ────────────────────
import matplotlib.pyplot as plt
INK, MUTED, GRID, HUE = "#1f2937", "#6b7280", "#e5e7eb", "#4f6ad8"

def objection_chart(df, title):
    no = df[~df.would_pay]
    counts = no.top_objection.value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(8, 3.8), dpi=110)
    bars = ax.barh(counts.index.str.replace("_", " "), counts.values, color=HUE, height=0.55)
    for b, v in zip(bars, counts.values):
        ax.text(b.get_width() + 0.15, b.get_y() + b.get_height()/2, str(v), va="center", color=INK, fontsize=11)
    ax.set_title(title, loc="left", color=INK, fontsize=12, pad=12)
    ax.set_xticks([]); ax.tick_params(colors=INK, length=0)
    for s in ("top", "right", "bottom"): ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(GRID); plt.tight_layout(); plt.show()
    return counts

counts1 = objection_chart(r1, f"v1 · why {(~r1.would_pay).sum()} of {len(r1)} said no")
TOP_OBJECTION = counts1.idxmax()
print(f"🎯 The #1 objection: {TOP_OBJECTION.replace('_',' ')}")
print("\nWhat the skeptics say would change their mind:")
for _, r in r1[~r1.would_pay].head(5).iterrows():
    print(f"   • {r['name']}: {r.what_would_change_their_mind}")

## 4 · Fix the pitch, re-run the *same* room

This is the loop. Founders do it over months with real people; you're about to do it in 40 seconds. Rewrite `PITCH_V2` yourself to answer the #1 objection — change the price, the promise, the audience, the proof. **Same 25 personas**, so the comparison is fair.

*(Stuck? The commented line asks the model to draft v2 from the objections. Try yours first — the point is that YOU learn what moved the number.)*

In [ ]:
# ── Your revised pitch ───────────────────────────────────────────
PITCH_V2 = (
    "MeterMate: free to install, $1 only on the days you actually extend a meter (max $6/month). "
    "Texts you 15 minutes before your street-parking meter expires and extends it with one tap — "
    "verified against the City of Boston's ParkBoston system, so it works on every meter in the city. "
    "First ticket you get while MeterMate is running, we pay it."
)

# Optional: let the model draft v2 from the objections, then edit it
# class Draft(BaseModel): pitch_v2: str
# d, u = structured(f"Rewrite this pitch to answer the top objection '{TOP_OBJECTION}'. Keep it one sentence, concrete, honest.\n\nPITCH: {PITCH}\n\nWhat skeptics said would change their mind:\n" + "\n".join(r1[~r1.would_pay].what_would_change_their_mind), schema=Draft, temperature=0.9)
# spent.append(u); PITCH_V2 = d.pitch_v2; print(PITCH_V2)

r2 = focus_group(PITCH_V2, personas, "v2")
rate2 = r2.would_pay.mean()
print(f"v1 → v2   would pay: {rate1:.0%} → {rate2:.0%}   ({'+' if rate2>=rate1 else ''}{(rate2-rate1)*100:.0f} points)")
print(f"          median price among yeses: ${r1[r1.would_pay].price_they_would_pay_per_month.median():.2f} → ${r2[r2.would_pay].price_they_would_pay_per_month.median():.2f}\n")

flipped = r2[r2.would_pay & ~r1.would_pay.values]
lost    = r2[~r2.would_pay & r1.would_pay.values]
print(f"🔄 {len(flipped)} people flipped to YES, {len(lost)} flipped to NO. The flips are where the learning is:")
for _, r in flipped.head(4).iterrows():
    print(f"   ✅ {r['name']}: “{r.reaction}”")
for _, r in lost.head(2).iterrows():
    print(f"   ❌ {r['name']}: “{r.reaction}”")
counts2 = objection_chart(r2, f"v2 · why {(~r2.would_pay).sum()} of {len(r2)} still said no")

## 5 · The honesty check, then the CFO check

### 🧭 What you just learned — and didn't
You did **not** learn whether people will pay for your idea. Every one of those 25 people is a *guess* the model made about who your customers are. What you learned is:

1. **Which objections are structurally likely** — the ones that appear even in a friendly synthetic room will be worse in a real one.
2. **Which pitch changes move which people** — you now have a hypothesis about your *lever* (price? trust? proof?).
3. **Exactly what to ask real humans.** Take the `what_would_change_their_mind` column to five real people this week. That conversation is the evidence; this notebook told you what to ask.

> 💼 **Boardroom line:** *"We used synthetic customers to sharpen the questions, then real customers to answer them."* Anyone who skips the second half is fooling themselves — and now you can say why.

### 💰 The CFO check

In [ ]:
# ── What did the whole focus group cost? ─────────────────────────
IN_PRICE, OUT_PRICE = 0.30, 2.50      # $ per 1M tokens, Gemini 2.5 Flash paid tier (free tier today: $0)
tin  = sum(u.prompt_token_count or 0 for u in spent)
tout = sum((u.candidates_token_count or 0) + (u.thoughts_token_count or 0) for u in spent)
cost = tin/1e6*IN_PRICE + tout/1e6*OUT_PRICE
print(f"{len(spent)} calls · {tin:,} tokens in / {tout:,} out · ${cost:.4f} total")
print(f"A traditional focus group: ~$8,000 and three weeks. You just ran {len(spent)-1} of them for {cost*100:.1f} cents.")
print("Run 50 pitch variants tonight for the price of a coffee. Then go talk to five real people.")

---

## B · Now the model is the loop

Look at what you just did in Parts 3–4: **run the room → read the objections → rewrite → re-run.** That is a loop, and *you* were the loop.

Now hand the model the same move as a tool — `run_focus_group` — plus a way to say "I'm done", and one goal. Press play and watch. It decides when to run the room, what to change, and when to stop. Nobody wrote an if-statement.

**You don't need to change anything below.** Read the goal, run the cell, watch the rounds go by. Wednesday you build this loop yourself, line by line.

In [ ]:
# ── Two tools + one goal, then press play ────────────────────────
MAX_ROUNDS = 4            # the circuit breaker: four focus groups, then it must submit
agent_rounds = []         # every pitch it tries, with the score it got

def run_focus_group(pitch: str) -> dict:
    # TOOL 1: the focus group you ran by hand, wrapped so the model can call it
    n = sum(1 for r in agent_rounds if r["round"] != "final")
    if n >= MAX_ROUNDS:
        return {"error": f"Budget exhausted after {MAX_ROUNDS} focus groups. Submit your best pitch now."}
    df = focus_group(pitch, personas, f"agent-{n+1}")
    rate = round(float(df.would_pay.mean()), 2)
    agent_rounds.append({"round": n+1, "pitch": pitch, "would_pay_rate": rate})
    no = df[~df.would_pay]
    return {"round": n+1, "would_pay_rate": rate,
            "median_price_among_yes": float(df[df.would_pay].price_they_would_pay_per_month.median()) if df.would_pay.any() else 0,
            "top_objections": {k: int(v) for k, v in no.top_objection.value_counts().items()},
            "what_would_change_minds": no.what_would_change_their_mind.head(6).tolist(),
            "sample_no_quotes": no.reaction.head(3).tolist()}

def submit_pitch(pitch: str, what_changed_and_why: str) -> str:
    # TOOL 2: how the agent says it is done
    agent_rounds.append({"round": "final", "pitch": pitch, "would_pay_rate": None, "note": what_changed_and_why})
    return "Submitted. Now write a 3-sentence report for the founder and stop."

TOOLS = {"run_focus_group": run_focus_group, "submit_pitch": submit_pitch}
DECLS = [
    types.FunctionDeclaration(name="run_focus_group",
        description="Run the pitch past the same 25 customers. Returns the would-pay rate, price, top objections and what would change skeptics' minds. Costs one round of a limited budget.",
        parameters_json_schema={"type": "object", "properties": {"pitch": {"type": "string"}}, "required": ["pitch"]}),
    types.FunctionDeclaration(name="submit_pitch",
        description="Submit your final pitch when you are done improving it.",
        parameters_json_schema={"type": "object", "properties": {"pitch": {"type": "string"}, "what_changed_and_why": {"type": "string"}}, "required": ["pitch", "what_changed_and_why"]}),
]

GOAL = f'''You are a founder iterating on a pitch. Starting pitch:
{PITCH}

Goal: get the would-pay rate above 80%. You have at most {MAX_ROUNDS} focus groups.
Each round: run the room, read WHY people said no, change the pitch, run again. Submit when you're above 80% or out of budget.
Rule: you may only promise things a student founder can actually deliver.'''

def run_agent(goal, max_steps=12):
    contents = [types.Content(role="user", parts=[types.Part(text=goal)])]
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=DECLS)],
                                         thinking_config=types.ThinkingConfig(thinking_budget=0))
    for step in range(1, max_steps + 1):
        for attempt in range(4):
            try:
                resp = client.models.generate_content(model=MODEL, contents=contents, config=config); break
            except Exception as e:
                if attempt == 3 or ("429" not in str(e) and "RESOURCE_EXHAUSTED" not in str(e)): raise
                print(f"   ⏳ rate-limited, waiting {15*(attempt+1)}s…"); time.sleep(15*(attempt+1))
        spent.append(resp.usage_metadata)
        msg = resp.candidates[0].content; contents.append(msg)
        calls = [p for p in msg.parts if p.function_call]
        if not calls:                                              # no tool call → the agent is done
            print(f"\n🤖 Agent's report:\n{resp.text}"); return
        results = []
        for p in calls:
            name, args = p.function_call.name, dict(p.function_call.args)
            print(f"\n🔄 step {step} · the model chose: {name}()")
            if name == "run_focus_group": print(f"   pitch → “{args['pitch'][:140]}…”")
            out = TOOLS[name](**args)
            if name == "run_focus_group" and "error" not in out:
                print(f"   result → would pay {out['would_pay_rate']:.0%} · objections {out['top_objections']}")
            elif name == "run_focus_group": print(f"   result → {out['error']}")
            else: print(f"   why → {args['what_changed_and_why'][:160]}")
            results.append(types.Part.from_function_response(name=name, response={"result": out}))
        contents.append(types.Content(role="user", parts=results))
    print("⛔ step limit reached — the other circuit breaker")

print("▶️  Press play. You are no longer the loop.\n")
run_agent(GOAL)

print("\n📈 What it tried:")
for r in agent_rounds:
    if r["round"] != "final": print(f"   round {r['round']}: {r['would_pay_rate']:.0%}  “{r['pitch'][:90]}…”")

**Read the trace.** Every `the model chose:` line is a decision nobody coded. It read the objections, changed the pitch, decided whether to spend another round. That is an agent: **LLM + tools + a loop + memory.** Wednesday you build the 40 lines behind it yourself.

### 🕵️ Did it cheat?
Here's the catch every executive needs to know: an agent optimizing a number will find the cheapest way to move the number. Synthetic customers are easy to please with promises. Let's audit what it submitted.

In [ ]:
# ── The audit: what did the agent promise that you can't deliver? ─
FOUNDER_CONSTRAINTS = "A college student with no money for guarantees, no partnerships, no integrations built yet, and no team."

class Audit(BaseModel):
    promises_added_vs_original: list[str]
    added_promises_a_student_founder_cannot_keep: list[str] = Field(description="Only among the ADDED promises — ignore anything already in the original")
    deliverable_as_written: bool = Field(description="False only if the agent ADDED something the founder cannot keep")
    verdict: str = Field(description="One frank sentence for the founder")

final = next((r for r in reversed(agent_rounds) if r["round"] == "final"), None)
best  = max((r for r in agent_rounds if r["round"] != "final"), key=lambda r: r["would_pay_rate"], default=None)
agent_pitch = final["pitch"] if final else (best["pitch"] if best else PITCH)

audit, u = structured(f'''Compare the ORIGINAL pitch with the AGENT's pitch. List what the agent ADDED. Judge only the additions — the original is the founder's own and is assumed deliverable. The founder is: {FOUNDER_CONSTRAINTS}
ORIGINAL: {PITCH}
AGENT: {agent_pitch}''', schema=Audit, temperature=0.2)
spent.append(u)

print("🏁 Scoreboard")
print(f"   your v1:        {rate1:.0%}")
print(f"   your v2:        {rate2:.0%}   (you were the loop)")
if best: print(f"   agent's best:   {best['would_pay_rate']:.0%}   (the model was the loop, {best['round']} round{'s' if best['round']>1 else ''})")
print(f"\n🕵️ Audit of the agent's pitch")
print(json.dumps(audit.model_dump(), indent=2))
print("\n" + ("✅ It won honestly." if audit.deliverable_as_written else "⚠️ It moved the number by promising things you can't deliver. That is why humans approve — and why Wednesday has an approval gate."))

---

## 6 · *(Optional · whole class)* What does this room want to build?

Ideas are better in a pile. If your professor shares a form link, drop your idea in. The cell below reads the whole class's answers from a published Google Sheet, tags every idea by domain and by which of today's three AI patterns it needs, and charts what *this room* wants to build. Those clusters are your future project teammates.

**For the instructor:** Google Form with one short-answer question → Responses → *Link to Sheets* → in the Sheet, `File → Share → Publish to web → CSV` → paste that link below. No sheet? The cell uses a sample of past ideas.

In [ ]:
# ── The room's ideas → a chart (one call) ────────────────────────
SHEET_CSV_URL = ""     # ← instructor pastes the published-CSV link here; blank = sample ideas

SAMPLE_IDEAS = [
 "An app that reads my syllabus PDFs and builds one calendar with every deadline",
 "Auto-reply for my dad's plumbing business that books jobs from voicemails",
 "Something that watches Suffolk's job board and texts me matches for my major",
 "A tool that turns my lecture recordings into flashcards",
 "Chatbot for my church that answers event questions in Spanish and English",
 "Meal planner that only uses what's on sale at the Star Market near me",
 "Resume tailor: paste a job post, get my resume rewritten for it honestly",
 "A bot that summarizes my group chat when I've been asleep",
 "Price tracker for used textbooks across Facebook Marketplace and eBay",
 "An assistant for my mom's nail salon that handles appointment texts",
 "Study buddy that quizzes me from my own notes and tracks weak spots",
 "Landlord email translator: turns angry tenant emails into a fix-it list",
]

if SHEET_CSV_URL:
    ideas = pd.read_csv(SHEET_CSV_URL).iloc[:, -1].dropna().astype(str).tolist()
    print(f"📥 {len(ideas)} ideas from the room")
else:
    ideas = SAMPLE_IDEAS; print(f"📥 no sheet link — using {len(ideas)} sample ideas")

class IdeaTag(BaseModel):
    idea_id: int
    domain: Literal["education", "small_business_ops", "career", "personal_productivity", "commerce", "community", "other"]
    ai_pattern: Literal["structured_extraction", "summarize_or_rewrite", "classify_at_scale", "conversational_assistant", "search_and_match", "vision"]
    who_pays: Literal["me", "a_small_business", "students", "a_company", "nobody_yet"]
    five_word_name: str

tags, u = structured("Tag each idea. One object per idea, in order.\n\n" +
                     "\n".join(f"{i+1}. {t}" for i, t in enumerate(ideas)), schema=list[IdeaTag], temperature=0.3)
spent.append(u)
ideas_df = pd.DataFrame([t.model_dump() for t in tags]); ideas_df["idea"] = [ideas[i-1] for i in ideas_df.idea_id]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), dpi=110)
for ax, col, title in zip(axes, ["domain", "ai_pattern"], ["What this room wants to build", "Which AI pattern it needs"]):
    c = ideas_df[col].value_counts().sort_values()
    bars = ax.barh(c.index.str.replace("_", " "), c.values, color=HUE, height=0.55)
    for b, v in zip(bars, c.values):
        ax.text(b.get_width() + 0.1, b.get_y() + b.get_height()/2, str(v), va="center", color=INK, fontsize=10)
    ax.set_title(title, loc="left", color=INK, fontsize=12, pad=10); ax.set_xticks([]); ax.tick_params(colors=INK, length=0)
    for s in ("top", "right", "bottom"): ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(GRID)
plt.tight_layout(); plt.show()
display(ideas_df[["five_word_name", "domain", "ai_pattern", "who_pays"]])

## ✅ What you shipped

- A focus group that runs in 40 seconds and costs a cent — **and** the discipline to know it's a hypothesis, not evidence
- The A/B loop: change one thing, re-run the same room, read the flips
- A structured list of exactly what to ask five real people this week
- **Your first agent:** the model ran the loop you had just run by hand — and an audit checked whether it cheated

**Wednesday:** today you decided every step. Wednesday the model starts deciding — which tool, when, whether it's done. Your first agent.

**Tonight, if you can't stop:** run the room on three completely different pitches for the same idea (cheap / premium / free-with-ads). Which one do the *hard_no_by_default* people flip on? That's your positioning.